<a href="https://colab.research.google.com/github/ehhkarinn/polish-accent-classifier/blob/main/accent_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade --force-reinstall tensorflow==2.19.0 tensorflow-text==2.19.0 tf-keras==2.19.0 tensorflow-decision-forests==1.4.0


ERROR: Could not find a version that satisfies the requirement tensorflow-decision-forests==1.4.0 (from versions: 1.8.1, 1.12.0)
ERROR: No matching distribution found for tensorflow-decision-forests==1.4.0


In [ ]:
import random
import numpy as np
import tensorflow as tf
import os

seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

os.environ['TF_DETERMINISTIC_OPS'] = '1'


In [ ]:

!git clone https://github.com/k-farruh/speech-accent-detection.git
%cd speech-accent-detection


Cloning into 'speech-accent-detection'...
remote: Enumerating objects: 219, done.
remote: Counting objects: 100% (219/219), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 219 (delta 86), reused 198 (delta 73), pack-reused 0 (from 0)
Receiving objects: 100% (219/219), 22.20 MiB | 21.83 MiB/s, done.
Resolving deltas: 100% (86/86), done.
/content/speech-accent-detection


In [ ]:

!pip install numpy pandas matplotlib librosa tensorflow scikit-learn


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
data_path = '/content/drive/MyDrive/movie_samples/'



In [ ]:
import pandas as pd
df = pd.read_csv(data_path + 'accent_labels.csv')


In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])

num_classes = len(label_encoder.classes_)
y = to_categorical(df['label_encoded'], num_classes=num_classes)


In [ ]:
import librosa
import numpy as np

def extract_ffnn_features_fixed(path):
    y, sr = librosa.load(path, sr=16000)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    if mfcc.shape[1] < 57:
        pad_width = 57 - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)), mode='constant')
    else:
        mfcc = mfcc[:, :57]

    flat = mfcc.T.flatten()[:738]
    return flat.reshape(1, -1)


In [ ]:
import os
X = []
valid_labels = []

for i, row in df.iterrows():
    file_path = os.path.join(data_path, row['filename'])
    try:
        features = extract_ffnn_features_fixed(file_path)
        X.append(features[0])
        valid_labels.append(row['label_encoded'])
    except Exception as e:
        print(f"Skipping {row['filename']}: {e}")

import numpy as np
X = np.array(X)
y = to_categorical(np.array(valid_labels), num_classes=num_classes)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout, Input

ffnn_model = Sequential([
    Input(shape=(738,)),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(len(np.unique(valid_labels)), activation='softmax')
])

ffnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

ffnn_model.fit(X_train, y_train, epochs=30, batch_size=16, validation_data=(X_val, y_val))


Epoch 1/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 340ms/step - accuracy: 0.0938 - loss: 3.0154 - val_accuracy: 0.1250 - val_loss: 24.8435
Epoch 2/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.4062 - loss: 1.8244 - val_accuracy: 0.1250 - val_loss: 14.9816
Epoch 3/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.5625 - loss: 1.1858 - val_accuracy: 0.1250 - val_loss: 11.3959
Epoch 4/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.6875 - loss: 0.8478 - val_accuracy: 0.1250 - val_loss: 9.1717
Epoch 5/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.8750 - loss: 0.4924 - val_accuracy: 0.1250 - val_loss: 7.9161
Epoch 6/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.8750 - loss: 0.4363 - val_accuracy: 0.1250 - val_loss: 7.5007
Epoch 7/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.8750 - loss: 0.5560 - val_accuracy: 0.1250 - val_loss: 7.2076
Epoch 8/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.9688 - loss: 0.2533 - val_accuracy: 0.1250 - val_loss: 6.

In [ ]:
ffnn_model.save('/content/drive/MyDrive/trained_accent_ffnn.h5')


In [ ]:
from tensorflow.keras.models import load_model

model = load_model('/content/drive/MyDrive/trained_accent_ffnn.h5')


In [ ]:

labels = label_encoder.classes_


In [ ]:
import os
movie_folder = '/content/drive/MyDrive/movie_samples/fake_polish'
movie_files = [f for f in os.listdir(movie_folder) if f.endswith('.wav')]

for fname in movie_files:
    fpath = os.path.join(movie_folder, fname)
    try:
        features = extract_ffnn_features_fixed(fpath)
        prediction = model.predict(features)
        predicted_label = labels[prediction.argmax()]
        confidence = prediction.max()

        print(f"File: {fname}")
        print(f"Predicted Accent: {predicted_label} (Confidence: {confidence:.2f})")
    except Exception as e:
        print(f"Error with {fname}: {e}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
File: defiance_01.wav
Predicted Accent: Polish (Confidence: 0.83)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
File: defiance_02.wav
Predicted Accent: Ukrainian (Confidence: 1.00)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
File: sophies_choice_01.wav
Predicted Accent: Polish (Confidence: 0.37)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
File: zoo_01.wav
Predicted Accent: Polish (Confidence: 0.98)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
File: zoo_02.wav
Predicted Accent: Ukrainian (Confidence: 0.94)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
File: immigrant_01.wav
Predicted Accent: Polish (Confidence: 0.55)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
File: immigrant_02.wav
Predicted Accent: Polish (Confidence: 0.98)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
File: sophies_choice_02.wav
Predicted Accent: Ukrainian (Confidence: 0.69)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
File: sophies_choice_03.wav
Predicted Accent: British (Confidence: 0.88)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
File: 